# Topic 2: ETL Design Patterns

> **Phase 5 → Week 9 → Topic 2**

---

## Common Production ETL Patterns

```
1. SCD Type 1  — overwrite old value with new (no history)
2. SCD Type 2  — track full history with start/end dates + is_current flag
3. Deduplication  — keep one canonical record per business key
4. Slowly Changing Lookup  — broadcast-able reference tables with cache invalidation
5. Late Arriving Facts  — handle events that arrive after their time window closes
6. Idempotent ETL  — run the same job twice → same result (no duplicates, no gaps)
```

---

## Interview Cheat Sheet

**Q: What is SCD Type 2 and how do you implement it in Spark?**
> SCD (Slowly Changing Dimension) Type 2 tracks full history: when a dimension attribute changes (e.g., customer moves city), instead of overwriting, you close the old row (`end_date = today, is_current = false`) and insert a new row (`start_date = today, end_date = null, is_current = true`). In Spark: use Delta MERGE with a `whenMatchedUpdate` (close old row) + `whenNotMatched` pattern, or a two-pass window function approach on plain Parquet.

**Q: What does idempotent ETL mean and why is it important?**
> Idempotent means running the pipeline multiple times with the same input produces the same output — no duplicates, no data loss. This is critical because pipelines fail and get re-run (by Airflow retry, manual re-trigger, etc.). Key techniques: (1) use `INSERT OVERWRITE` for partition-based loads (not append), (2) use Delta MERGE (upsert) which is naturally idempotent, (3) always dedup on the primary key after loading.

**Q: What is the medallion architecture?**
> A layered data lake pattern with three zones: **Bronze** (raw ingestion — original data as-is, append-only, no transformations, full history), **Silver** (cleaned & validated — bad records removed, schema enforced, deduped, joined with dimensions), **Gold** (aggregated & business-ready — KPIs, summaries, ML features, optimized for consumption). Each layer is typically stored as Delta tables.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import shutil, os

spark = SparkSession.builder \
    .appName("Week9 - ETL Design Patterns") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: SCD Type 1 — Overwrite

In [ ]:
# SCD Type 1: update in place — no history retained
# Use case: correcting errors, updating non-historical attributes

SCD1_PATH = "/tmp/scd1_customers"
if os.path.exists(SCD1_PATH): shutil.rmtree(SCD1_PATH)

# Initial customer dimension
customers_v1 = spark.createDataFrame([
    ("C001", "Alice", "New York",  "Gold"),
    ("C002", "Bob",   "Chicago",   "Silver"),
    ("C003", "Carol", "Houston",   "Bronze"),
], ["customer_id", "name", "city", "tier"])

customers_v1.write.format("delta").save(SCD1_PATH)
print("Initial customer dimension:")
spark.read.format("delta").load(SCD1_PATH).show()

# Incoming update: Alice moved city, Bob upgraded tier
updates = spark.createDataFrame([
    ("C001", "Alice", "Los Angeles", "Gold"),    # city changed
    ("C002", "Bob",   "Chicago",     "Gold"),    # tier upgraded
    ("C004", "Dave",  "Seattle",     "Bronze"),  # new customer
], ["customer_id", "name", "city", "tier"])

from delta.tables import DeltaTable
scd1_table = DeltaTable.forPath(spark, SCD1_PATH)

# SCD1 MERGE: just update all columns in place
scd1_table.alias("t") \
  .merge(updates.alias("s"), "s.customer_id = t.customer_id") \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

print("After SCD1 update (history lost — only current state):")
spark.read.format("delta").load(SCD1_PATH).orderBy("customer_id").show()

## Part 2: SCD Type 2 — Full History

In [ ]:
# SCD Type 2: keep full history with start_date, end_date, is_current
# Use case: customer address changes, price changes, org restructuring

SCD2_PATH = "/tmp/scd2_customers"
if os.path.exists(SCD2_PATH): shutil.rmtree(SCD2_PATH)

from datetime import date

# Initial load with SCD2 metadata columns
customers_scd2 = spark.createDataFrame([
    ("C001", "Alice", "New York",  "Gold",   date(2023, 1, 1), None,  True),
    ("C002", "Bob",   "Chicago",   "Silver", date(2023, 1, 1), None,  True),
], ["customer_id", "name", "city", "tier", "start_date", "end_date", "is_current"])

customers_scd2.write.format("delta").save(SCD2_PATH)
print("Initial SCD2 dimension:")
spark.read.format("delta").load(SCD2_PATH).show()

# Day-2: Alice moved to LA, Dave is a new customer
def apply_scd2(target_path, incoming_df, key_col, change_cols, effective_date):
    """
    Apply SCD Type 2 logic:
    - Close old row (set end_date, is_current=False) for changed records
    - Insert new current row for changed records
    - Insert new row for truly new records
    """
    target = DeltaTable.forPath(spark, target_path)
    today = F.lit(effective_date)

    # Detect what actually changed (compare change_cols)
    change_condition = " OR ".join(
        [f"s.{c} != t.{c}" for c in change_cols]
    )

    # Step 1: Close old rows where something changed
    target.alias("t").merge(
        incoming_df.alias("s"),
        f"s.{key_col} = t.{key_col} AND t.is_current = true"
    ).whenMatchedUpdate(
        condition=change_condition,
        set={"end_date": today, "is_current": F.lit(False)}
    ).execute()

    # Step 2: Insert new rows (changed + new)
    existing_current = spark.read.format("delta").load(target_path) \
        .filter(F.col("is_current")) \
        .select(key_col)

    # New rows = incoming rows that are NEW or CHANGED
    closed = spark.read.format("delta").load(target_path) \
        .filter(F.col("end_date") == effective_date)  # just closed today

    rows_to_insert = incoming_df.join(
        existing_current, on=key_col, how="left"
    )  # include ALL incoming (new + changed)

    new_rows = incoming_df \
        .withColumn("start_date", today) \
        .withColumn("end_date", F.lit(None).cast(DateType())) \
        .withColumn("is_current", F.lit(True))

    new_rows.write.format("delta").mode("append").save(target_path)

incoming_day2 = spark.createDataFrame([
    ("C001", "Alice", "Los Angeles", "Gold"),  # city changed
    ("C003", "Dave",  "Seattle",     "Bronze"), # new customer
], ["customer_id", "name", "city", "tier"])

apply_scd2(SCD2_PATH, incoming_day2, "customer_id", ["city", "tier"], date(2024, 2, 1))

print("SCD2 after Day-2 changes (Alice has 2 rows — history preserved):")
spark.read.format("delta").load(SCD2_PATH).orderBy("customer_id", "start_date").show()

## Part 3: Medallion Architecture

In [ ]:
# Medallion Architecture: Bronze → Silver → Gold
BRONZE = "/tmp/medallion/bronze/orders"
SILVER = "/tmp/medallion/silver/orders"
GOLD   = "/tmp/medallion/gold/revenue_by_category"

for p in [BRONZE, SILVER, GOLD]:
    if os.path.exists(p): shutil.rmtree(p)

# ── BRONZE: raw ingestion — as-is, append-only, full history ────────────────
raw_batch = spark.createDataFrame([
    ("O001", "C001", "P1",  250.0, "delivered", "2024-01-15", "2024-01-15T10:00:00"),
    ("O002", "C002", "P2",  -99.0, "pending",   "2024-01-16", "2024-01-16T11:00:00"),  # bad
    ("O003", None,   "P1",  500.0, "delivered", "2024-01-17", "2024-01-17T12:00:00"),  # null
    ("O001", "C001", "P1",  250.0, "delivered", "2024-01-15", "2024-01-15T10:00:00"),  # dup
], ["order_id", "customer_id", "product_id", "amount", "status", "order_date", "ingested_at"])

raw_batch.write.format("delta").mode("append").save(BRONZE)
print(f"Bronze: {spark.read.format('delta').load(BRONZE).count()} rows (raw, no cleaning)")

# ── SILVER: clean + validate + dedup ─────────────────────────────────────────
bronze_df = spark.read.format("delta").load(BRONZE)

silver_df = bronze_df \
    .filter(F.col("amount") > 0) \
    .filter(F.col("customer_id").isNotNull()) \
    .dropDuplicates(["order_id"]) \
    .withColumn("order_date", F.to_date("order_date"))

silver_df.write.format("delta").mode("overwrite").save(SILVER)
print(f"Silver: {spark.read.format('delta').load(SILVER).count()} rows (clean, deduped)")

# ── GOLD: aggregated business metrics ────────────────────────────────────────
products = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)

gold_df = silver_df \
    .join(F.broadcast(products.select("product_id", "category")), on="product_id", how="left") \
    .groupBy("category") \
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("*").alias("order_count"),
        F.avg("amount").alias("avg_order_value")
    ) \
    .orderBy(F.col("total_revenue").desc())

gold_df.write.format("delta").mode("overwrite").save(GOLD)
print("Gold: Revenue by category:")
spark.read.format("delta").load(GOLD).show()

## Part 4: Idempotent ETL

In [ ]:
# Idempotent = run it 10 times, always get the same result

IDEM_PATH = "/tmp/idempotent_output"
if os.path.exists(IDEM_PATH): shutil.rmtree(IDEM_PATH)

def run_daily_etl(batch_date: str):
    """Idempotent daily ETL: safe to re-run multiple times."""
    # Read source for this date
    batch = spark.createDataFrame([
        ("O001", 250.0, batch_date),
        ("O002",  89.5, batch_date),
    ], ["order_id", "amount", "order_date"])

    # IDEMPOTENT: overwrite the specific partition for this date
    # Re-running with same batch_date overwrites same partition → no duplicates
    batch.write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"order_date = '{batch_date}'") \
        .save(IDEM_PATH)

    count = spark.read.format("delta").load(IDEM_PATH).count()
    print(f"  After run for {batch_date}: {count} total rows")

print("Run 1 — Jan 15:")
run_daily_etl("2024-01-15")

print("Run 2 — Jan 16:")
run_daily_etl("2024-01-16")

print("Re-run Jan 15 (simulates retry — result is same, not doubled!):")
run_daily_etl("2024-01-15")

print("\nFinal state:")
spark.read.format("delta").load(IDEM_PATH).show()

## Part 5: Deduplication at Scale

In [ ]:
# Production deduplication: keep latest version per business key

events = spark.createDataFrame([
    ("E001", "U1", "purchase", 99.0,  "2024-01-15 10:00:00"),
    ("E001", "U1", "purchase", 99.0,  "2024-01-15 10:00:00"),  # exact dup
    ("E002", "U2", "view",     None,  "2024-01-15 11:00:00"),
    ("E001", "U1", "purchase", 99.0,  "2024-01-15 10:05:00"),  # same ID, later timestamp
    ("E003", "U1", "click",    None,  "2024-01-16 09:00:00"),
], ["event_id", "user_id", "event_type", "amount", "event_time"])

# Tier 1: remove exact duplicates cheaply
step1 = events.dropDuplicates()
print(f"After dropDuplicates (exact): {step1.count()} rows")

# Tier 2: keep latest by event_id (deterministic winner)
w = Window.partitionBy("event_id").orderBy(F.col("event_time").desc())
step2 = step1 \
    .withColumn("_rn", F.row_number().over(w)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn")

print(f"After keep-latest (row_number): {step2.count()} rows")
step2.show()
print("E001 has only one row (latest timestamp wins)")

## Exercises

1. Implement a full SCD Type 1 pipeline: load a customer dimension, apply an update batch, verify no history is kept.
2. Build a 3-layer medallion pipeline for the orders CSV: Bronze (raw append), Silver (clean + dedup), Gold (revenue by country).
3. Write an idempotent ETL function using `replaceWhere` for date-partitioned Delta tables. Prove it's idempotent by running it 3 times with the same date.
4. What is the difference between `mode('overwrite')` and `option('replaceWhere', ...)` on a Delta table?
5. A colleague says "SCD Type 2 is always better than Type 1 because it keeps history." When would you choose Type 1?

In [ ]:
# Exercise 4: overwrite vs replaceWhere
print("""
mode('overwrite') vs option('replaceWhere', ...):

mode('overwrite'):
  Deletes ALL data in the table, then writes new data.
  Equivalent to DROP + CREATE + INSERT.
  Use for full refreshes.

option('replaceWhere', "order_date = '2024-01-15'"):
  Deletes ONLY the rows matching the predicate.
  Then inserts the new rows.
  All other partitions/rows are untouched.
  Requires Delta Lake — Parquet doesn't have this.

Example:
  Table has 365 partitions (one per day).
  overwrite    → deletes all 365 partitions, rewrites 1  (DANGEROUS!)
  replaceWhere → deletes only the 1 matching partition, rewrites it (SAFE)

Exercise 5: When to choose SCD Type 1:
  Choose Type 1 when:
  ✓ Historical values have no analytical meaning (e.g., fixing a typo in a name)
  ✓ The dimension changes so frequently that tracking history is impractical
  ✓ Storage cost of full history is prohibitive
  ✓ Downstream reports only care about the CURRENT state
  ✓ You're correcting bad data, not tracking real-world change

  Choose Type 2 when:
  ✓ Historical analysis requires knowing which value was active at a given time
  ✓ 'Customer was in Gold tier when they made this purchase' matters
  ✓ Regulatory/audit requirements to track attribute history
""")